# Call Center Volume Trend Analysis

## Project Overview
Analyze call center data over time to understand peak periods, daily/weekly cycles, and staffing implications.

## Learning Objectives
- Identify hourly and daily call volume patterns
- Detect weekly and seasonal trends in call volume
- Analyze call type distribution over time
- Calculate staffing efficiency metrics
- Identify anomalous volume spikes

## Problem Statement
Call center managers need to understand volume patterns to right-size staffing. Under-staffing increases wait times and customer churn; over-staffing wastes payroll.

## Why This Project Matters
Labor costs are 60-70% of call center expenses. Accurate volume pattern analysis enables shift scheduling that balances service quality with cost efficiency.

## Dataset Overview
Synthetic hourly call center data: 2 years × 3 call types (Support, Billing, Sales) with realistic hourly/weekly/seasonal patterns.

## Dataset Source and License Notes
- Synthetic data
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n_days = 365 * 2
dates = pd.date_range('2022-01-01', periods=n_days, freq='D')
call_types = ['Support', 'Billing', 'Sales']
base_volumes = {'Support': 80, 'Billing': 40, 'Sales': 30}

# Hourly pattern (calls per hour, 8am-8pm peak)
hourly_pattern = np.array([
    0.05, 0.02, 0.01, 0.01, 0.02, 0.04, 0.08, 0.15,  # 0-7
    0.40, 0.70, 0.85, 0.95, 0.80, 0.90, 1.00, 0.85,  # 8-15
    0.65, 0.45, 0.30, 0.20, 0.12, 0.08, 0.06, 0.05   # 16-23
])

# Weekly pattern (Mon=0, Sun=6)
weekly_pattern = np.array([1.10, 1.15, 1.05, 1.00, 0.95, 0.50, 0.35])

rows = []
for day_idx, date in enumerate(dates):
    dow = date.dayofweek
    month = date.month
    # Seasonal: higher in Jan (New Year billing), Dec (holiday support)
    seasonal = 1 + 0.15 * np.cos(2 * np.pi * (month - 1) / 12)
    # Monday surge after weekends
    monday_surge = 1.2 if dow == 0 else 1.0
    for hour in range(24):
        for ctype in call_types:
            base = base_volumes[ctype]
            vol = base * hourly_pattern[hour] * weekly_pattern[dow] * seasonal * monday_surge
            # Add noise
            vol = max(0, int(np.random.poisson(max(1, vol))))
            rows.append({'DateTime': date + pd.Timedelta(hours=hour),
                         'Date': date, 'Hour': hour, 'DayOfWeek': date.day_name(),
                         'Month': month, 'Year': date.year,
                         'CallType': ctype, 'Calls': vol})

df = pd.DataFrame(rows)
print(f'Dataset: {df.shape}')
print(f'Total calls: {df["Calls"].sum():,}')
print(f'Date range: {df["Date"].min()} to {df["Date"].max()}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'\nCalls per type:')
print(df.groupby('CallType')['Calls'].sum())
print(f'\nDays in dataset: {df["Date"].nunique()}')

## Hourly Call Pattern

In [ ]:
hourly = df.groupby(['Hour', 'CallType'])['Calls'].mean().unstack()
fig, ax = plt.subplots(figsize=(12, 5))
hourly.plot(ax=ax, marker='o', markersize=4)
ax.set_title('Average Calls per Hour by Type')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Avg Calls')
ax.set_xticks(range(24))
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'hourly_pattern.png'), dpi=100, bbox_inches='tight')
plt.show()

## Day of Week Pattern

In [ ]:
dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
daily_total = df.groupby(['Date', 'DayOfWeek'])['Calls'].sum().reset_index()
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=daily_total, x='DayOfWeek', y='Calls', order=dow_order, ax=ax)
ax.set_title('Daily Call Volume by Day of Week')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'dow_pattern.png'), dpi=100, bbox_inches='tight')
plt.show()

## Monthly Trend

In [ ]:
monthly = df.groupby([df['Date'].dt.to_period('M'), 'CallType'])['Calls'].sum().unstack()
monthly.index = monthly.index.astype(str)
fig, ax = plt.subplots(figsize=(14, 5))
monthly.plot.bar(ax=ax, stacked=True, edgecolor='black', linewidth=0.3)
ax.set_title('Monthly Call Volume by Type')
ax.set_ylabel('Total Calls')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'monthly_trend.png'), dpi=100, bbox_inches='tight')
plt.show()

## Peak Hour Heatmap

In [ ]:
pivot = df.groupby(['DayOfWeek', 'Hour'])['Calls'].mean()
# Reindex by dow order
pivot = pivot.reindex(dow_order, level=0)
pivot_df = pivot.unstack(level=1)
fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(pivot_df, cmap='YlOrRd', ax=ax, annot=False)
ax.set_title('Average Calls: Day of Week × Hour')
ax.set_ylabel('Day')
ax.set_xlabel('Hour')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'peak_heatmap.png'), dpi=100, bbox_inches='tight')
plt.show()

## Anomalous Volume Days

In [ ]:
daily_vol = df.groupby('Date')['Calls'].sum()
mean_vol = daily_vol.mean()
std_vol = daily_vol.std()
anomalies = daily_vol[daily_vol > mean_vol + 2 * std_vol]
print(f'Days with anomalously high volume (>2σ): {len(anomalies)}')
if len(anomalies) > 0:
    print(anomalies.sort_values(ascending=False).head(10))

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(daily_vol.index, daily_vol.values, alpha=0.5, linewidth=0.5)
ax.axhline(mean_vol + 2 * std_vol, color='red', linestyle='--', label='+2σ threshold')
if len(anomalies) > 0:
    ax.scatter(anomalies.index, anomalies.values, color='red', s=15, zorder=5, label='Anomalies')
ax.set_title('Daily Call Volume with Anomaly Detection')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'anomalies.png'), dpi=100, bbox_inches='tight')
plt.show()

## Interpretation and Business Insight
- **Peak hours** are 10am-4pm with a lunch dip — staffing should ramp up from 8am and peak at 10am
- **Monday surge** is 15-20% above mid-week — Monday shifts need extra agents
- **Weekends** see 50-65% volume drop — reduced weekend teams are appropriate
- **Seasonal peaks** in December/January driven by holiday support and billing inquiries
- **Support calls** dominate volume — investment in self-service could reduce load
- Anomalous days likely correspond to outages, billing cycles, or promotions

## Limitations
- Synthetic data without real customer behavior complexity
- No average handle time or wait time data
- No agent performance or staffing count data
- No channel breakdown (phone vs chat vs email)

## How to Improve This Project
- Add handle time data for workload (Erlang C) modeling
- Include customer satisfaction scores per call
- Build staffing optimization model
- Add multi-channel analysis
- Include IVR containment rate data

## Production Considerations
- Real-time volume monitoring dashboards
- Automated shift scheduling from volume forecasts
- Alert system for anomalous volume spikes
- Integration with workforce management tools

## Common Mistakes
- Staffing to average volume instead of peak demand
- Ignoring Monday surge when scheduling
- Under-staffing weekends without checking if that's hurting metrics
- Not accounting for handle-time variation across call types

## Mini Challenge / Exercises
1. What is the busiest 3-hour window, and how much of daily volume does it capture?
2. Calculate the Monday-to-Friday volume ratio — is it consistent across call types?
3. If each agent handles 8 calls/hour, how many agents are needed at peak hour on Monday?
4. What percentage of total weekly calls come on weekends?

## Final Summary / Key Takeaways
- Call volume follows strong hourly, daily, and seasonal patterns
- Monday surge and mid-day peaks are the primary staffing drivers
- Weekend volume drops significantly — flexible staffing saves costs
- Seasonal patterns allow advance planning for high-volume periods
- Anomaly detection catches unexpected spikes early for reactive staffing